# EXERCISE 1: Vectorized Measurement Quality Analysis

In [134]:
# [CELL 1] Import Libraries

import numpy as np
import pandas as pd

In [135]:
# [CELL 2] Student Configuration

SEED_DIGIT = 6

RANDOM_SEED = SEED_DIGIT

print("SEED_DIGIT:", SEED_DIGIT)
print("RANDOM_SEED:", RANDOM_SEED)

SEED_DIGIT: 6
RANDOM_SEED: 6


In [136]:
# [CELL 3] Load SECOM Measurement Matrix

X = pd.read_csv(
    "secom.data",
    sep=r"\s+",
    header=None,
    na_values="NaN"
)
print("Dataset Shape:", X.shape)
display(X.head())

Dataset Shape: (1567, 590)


,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [137]:
# [CELL 4] Identify Missing and Constant Features

missing_values = X.isna().sum()

unique_values = X.nunique(dropna=True)

missing_features = X.columns[
    X.count() == 0
].tolist()

constant_features = X.columns[
    unique_values <= 1
].tolist()

usable_features = [
    col for col in X.columns
    if X[col].count() > 0
    and unique_values[col] > 1
]

print("Missing Features:", missing_features)
print("Constant Features:", constant_features)
print("Number of Usable Features:", len(usable_features))

Missing Features: []
Constant Features: [5, 13, 42, 49, 52, 69, 97, 141, 149, 178, 179, 186, 189, 190, 191, 192, 193, 194, 226, 229, 230, 231, 232, 233, 234, 235, 236, 237, 240, 241, 242, 243, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 276, 284, 313, 314, 315, 322, 325, 326, 327, 328, 329, 330, 364, 369, 370, 371, 372, 373, 374, 375, 378, 379, 380, 381, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 414, 422, 449, 450, 451, 458, 461, 462, 463, 464, 465, 466, 481, 498, 501, 502, 503, 504, 505, 506, 507, 508, 509, 512, 513, 514, 515, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538]
Number of Usable Features: 474


In [138]:
# [CELL 5] Select Usable Features

n_features = 5 + (SEED_DIGIT % 3)

rng = np.random.default_rng(RANDOM_SEED)

selected_features = sorted(
    rng.choice(
        usable_features,
        size=n_features,
        replace=False
    ).tolist()
)

selected_data = X[selected_features]

print("Number of Selected Features:", n_features)
print("Selected Features:", selected_features)

Number of Selected Features: 5
Selected Features: [171, 228, 289, 298, 564]


In [139]:
# [CELL 6] Calculate Feature-Wise Statistics

statistics = pd.DataFrame({
    "Mean": selected_data.mean(),
    "Median": selected_data.median(),
    "Sample_Std": selected_data.std(ddof=1),
    "Missing_Percentage": selected_data.isna().mean() * 100
})

display(statistics)

,Mean,Median,Sample_Std,Missing_Percentage
171,0.120064,0.1125,0.060785,0.063816
228,0.017844,0.0155,0.010745,0.127632
289,2.578596,2.5490,1.616993,0.191449
298,0.063804,0.0528,0.064225,0.127632
564,6.444985,5.1600,2.633583,17.421825


In [140]:
# [CELL 7] Calculate Median Absolute Deviation

medians = selected_data.median()

mad = selected_data.sub(
    medians,
    axis=1
).abs().median()

print("Feature Medians:")
display(medians)

print("Median Absolute Deviation:")
display(mad)

Feature Medians:


171    0.1125
228    0.0155
289    2.5490
298    0.0528
564    5.1600
dtype: float64

Median Absolute Deviation:


171    0.03180
228    0.00340
289    0.46665
298    0.01350
564    1.06000
dtype: float64

In [141]:
# [CELL 8] Calculate Robust Standardized Values

safe_mad = mad.replace(0, np.nan)

robust_z = (
    0.6745 *
    selected_data.sub(medians, axis=1)
    .div(safe_mad, axis=1)
)

# Handle features with zero MAD
for col in mad[mad == 0].index:

    robust_z[col] = np.where(
        selected_data[col].isna(),
        np.nan,
        np.where(
            selected_data[col] == medians[col],
            0,
            np.where(
                selected_data[col] > medians[col],
                np.inf,
                -np.inf
            )
        )
    )

display(robust_z.head())

,171,228,289,298,564
0,0.029695,7.756750,1.872816,-0.774426,NaN
1,1.679887,1.606897,1.784502,-0.129904,NaN
2,-1.603528,0.297574,0.369446,1.358993,-2.583462
3,-1.631102,-0.119029,2.010997,-1.214100,1.374453
4,-1.047808,2.321074,2.599423,1.134159,NaN


In [142]:
# [CELL 9] Identify Unusual Measurements

flag_mask = robust_z.abs() > 3.5

flagged_counts = flag_mask.sum()

observed_counts = selected_data.notna().sum()

flagged_percentages = (
    flagged_counts / observed_counts
) * 100

print("Flagged Counts:")
display(flagged_counts)

print("Flagged Percentages:")
display(flagged_percentages)

Flagged Counts:


171     47
228     62
289      2
298     40
564    109
dtype: int64

Flagged Percentages:


171    3.001277
228    3.961661
289    0.127877
298    2.555911
564    8.423493
dtype: float64

In [143]:
# [CELL 10] Rank Features and Export CSV

statistics["Flagged_Count"] = flagged_counts

statistics["Flagged_Percentage"] = flagged_percentages

statistics = statistics.sort_values(
    by="Flagged_Percentage",
    ascending=False
)

statistics.to_csv(
    "measurement_quality.csv",
    index_label="Feature"
)

print("Ranked Measurement Quality Results:")

display(statistics)

print("CSV file successfully generated!")

Ranked Measurement Quality Results:


,Mean,Median,Sample_Std,Missing_Percentage,Flagged_Count,Flagged_Percentage
564,6.444985,5.1600,2.633583,17.421825,109,8.423493
228,0.017844,0.0155,0.010745,0.127632,62,3.961661
171,0.120064,0.1125,0.060785,0.063816,47,3.001277
298,0.063804,0.0528,0.064225,0.127632,40,2.555911
289,2.578596,2.5490,1.616993,0.191449,2,0.127877


CSV file successfully generated!


In [144]:
# [CELL 11] Generate Assessment Data

highest_missing_feature = (
    statistics["Missing_Percentage"].idxmax()
)

highest_flagged_feature = (
    statistics["Flagged_Percentage"].idxmax()
)

total_flagged_observations = int(
    flag_mask.to_numpy().sum()
)

vectorized_verified = (
    flag_mask.shape == selected_data.shape
    and statistics.shape[0] == len(selected_features)
)

assessment = pd.DataFrame({
    "Output Parameter": [
        "Selected Features",
        "Highest Missing Percentage",
        "Feature with Highest Flagged Percentage",
        "Total Flagged Observations",
        "Vectorized Results Verified"
    ],

    "Observed Value": [
        ", ".join(map(str, selected_features)),
        highest_missing_feature,
        highest_flagged_feature,
        total_flagged_observations,
        vectorized_verified
    ]
})

display(assessment)

,Output Parameter,Observed Value
0,Selected Features,"171, 228, 289, 298, 564"
1,Highest Missing Percentage,564
2,Feature with Highest Flagged Percentage,564
3,Total Flagged Observations,260
4,Vectorized Results Verified,True


# EXERCISE 2: Mobile Device Usage Profiling

In [145]:

# [CELL 1] Load SECOM Labels and Preserve Original Record IDs

labels_raw = pd.read_csv(
    "secom_labels.data",
    sep=r"\s+",
    header=None
)

labels = labels_raw.iloc[:, 0].astype(int)

if len(X) != len(labels):
    raise ValueError("Measurement and label row counts do not match!")

df = X.copy()

# Preserve original row identifiers
df.insert(0, "Record_ID", np.arange(len(df)))

# Assign Pass and Fail labels
df["Outcome"] = labels.map({
    1: "Pass",
    -1: "Fail"
})

# Remove records with unrecognized labels, if any
df = df.dropna(subset=["Outcome"]).copy()

feature_cols = X.columns.tolist()

print("Dataset Shape:", df.shape)
print("\nOutcome Counts:")
print(df["Outcome"].value_counts())

display(df.head())

Dataset Shape: (1567, 592)

Outcome Counts:
Outcome
Fail    1463
Pass     104
Name: count, dtype: int64


,Record_ID,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Outcome
0,0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,Fail
1,1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,Fail
2,2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,Pass
3,3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,Fail
4,4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,Fail


In [146]:
# [CELL 2] Create Balanced Sample

ID_SUM = 0  # Replace with your assigned ID_SUM

pass_count = (df["Outcome"] == "Pass").sum()
fail_count = (df["Outcome"] == "Fail").sum()

n_per_group = min(
    20 + ID_SUM,
    pass_count,
    fail_count
)

if n_per_group < 2:
    raise ValueError("Insufficient records for balanced sampling.")

rng = np.random.default_rng(RANDOM_SEED)

pass_records = df[df["Outcome"] == "Pass"]
fail_records = df[df["Outcome"] == "Fail"]

pass_sample = pass_records.iloc[
    rng.choice(len(pass_records), n_per_group, replace=False)
]

fail_sample = fail_records.iloc[
    rng.choice(len(fail_records), n_per_group, replace=False)
]

balanced_sample = pd.concat(
    [pass_sample, fail_sample]
).sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Records per Outcome:")
print(balanced_sample["Outcome"].value_counts())

print("\nTotal Balanced Sample Size:", len(balanced_sample))

Records per Outcome:
Outcome
Pass    20
Fail    20
Name: count, dtype: int64

Total Balanced Sample Size: 40


In [147]:
# [CELL 3] Select Five Usable Measurement Features

usable_features = [
    col for col in feature_cols
    if df[col].notna().any()
    and df[col].nunique(dropna=True) > 1
]

feature_rng = np.random.default_rng(RANDOM_SEED)

selected_features = feature_rng.choice(
    usable_features,
    size=5,
    replace=False
).tolist()

selected_features.sort()

print("Selected Features:", selected_features)

Selected Features: [171, 228, 289, 298, 564]


In [148]:
# [CELL 4] Apply Median Imputation

imputation_medians = balanced_sample[
    selected_features
].median()

balanced_sample.loc[:, selected_features] = (
    balanced_sample[selected_features].fillna(imputation_medians)
)

print("Imputation Medians:")
print(imputation_medians)

print("\nRemaining Missing Values:")
print(balanced_sample[selected_features].isna().sum())

Imputation Medians:
171    0.10735
228    0.01500
289    2.54005
298    0.05960
564    4.98000
dtype: float64

Remaining Missing Values:
171    0
228    0
289    0
298    0
564    0
dtype: int64


In [149]:
# [CELL 5] Calculate Group Statistics

group_statistics = balanced_sample.groupby("Outcome")[
    selected_features
].agg(["count", "mean", "median", "std"])

display(group_statistics)

171                                228                              \
        count      mean   median       std count     mean   median       std   
Outcome                                                                        
Fail       20  0.134420  0.11215  0.078216    20  0.02082  0.01265  0.020434   
Pass       20  0.098805  0.09770  0.039049    20  0.01619  0.01570  0.004800   

          289                                298                               \
        count      mean   median       std count      mean   median       std   
Outcome                                                                         
Fail       20  2.460825  2.54005  0.457526    20  0.058070  0.05165  0.025055   
Pass       20  2.625775  2.55615  0.690224    20  0.083785  0.06335  0.086414   

          564                           
        count    mean median       std  
Outcome                                 
Fail       20  6.2370   4.98  2.874602  
Pass       20  6.0115   4.98  2.656823

In [150]:
# [CELL 6] Calculate Mean Differences and Standardized Mean Differences

pass_data = balanced_sample[
    balanced_sample["Outcome"] == "Pass"
][selected_features]

fail_data = balanced_sample[
    balanced_sample["Outcome"] == "Fail"
][selected_features]

pass_mean = pass_data.mean()
fail_mean = fail_data.mean()

pass_std = pass_data.std(ddof=1)
fail_std = fail_data.std(ddof=1)

n_pass = len(pass_data)
n_fail = len(fail_data)

# Absolute Mean Difference
absolute_mean_difference = (fail_mean - pass_mean).abs()

# Pooled Within-Group Standard Deviation
pooled_variance = (
    (n_pass - 1) * pass_std**2
    + (n_fail - 1) * fail_std**2
) / (n_pass + n_fail - 2)

pooled_std = np.sqrt(pooled_variance)

# Handle zero denominators safely
standardized_mean_difference = (
    (fail_mean - pass_mean) / pooled_std.replace(0, np.nan)
)

print("Absolute Mean Difference:")
print(absolute_mean_difference)

print("\nStandardized Mean Difference:")
print(standardized_mean_difference)

Absolute Mean Difference:
171    0.035615
228    0.004630
289    0.164950
298    0.025715
564    0.225500
dtype: float64

Standardized Mean Difference:
171    0.576139
228    0.311940
289   -0.281701
298   -0.404196
564    0.081471
dtype: float64


In [151]:
# [CELL 7] Rank Features

feature_comparison = pd.DataFrame({
    "Pass_Count": pass_data.count(),
    "Fail_Count": fail_data.count(),
    "Pass_Mean": pass_mean,
    "Fail_Mean": fail_mean,
    "Pass_Median": pass_data.median(),
    "Fail_Median": fail_data.median(),
    "Pass_Std": pass_std,
    "Fail_Std": fail_std,
    "Absolute_Mean_Difference": absolute_mean_difference,
    "Pooled_Standard_Deviation": pooled_std,
    "Standardized_Mean_Difference": standardized_mean_difference,
    "Absolute_Standardized_Mean_Difference":
        standardized_mean_difference.abs()
})

feature_comparison = feature_comparison.sort_values(
    by="Absolute_Standardized_Mean_Difference",
    ascending=False,
    na_position="last"
)

display(feature_comparison)

,Pass_Count,Fail_Count,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Std,Fail_Std,Absolute_Mean_Difference,Pooled_Standard_Deviation,Standardized_Mean_Difference,Absolute_Standardized_Mean_Difference
171,20,20,0.098805,0.134420,0.09770,0.11215,0.039049,0.078216,0.035615,0.061817,0.576139,0.576139
298,20,20,0.083785,0.058070,0.06335,0.05165,0.086414,0.025055,0.025715,0.063620,-0.404196,0.404196
228,20,20,0.016190,0.020820,0.01570,0.01265,0.004800,0.020434,0.004630,0.014843,0.311940,0.311940
289,20,20,2.625775,2.460825,2.55615,2.54005,0.690224,0.457526,0.164950,0.585551,-0.281701,0.281701
564,20,20,6.011500,6.237000,4.98000,4.98000,2.656823,2.874602,0.225500,2.767855,0.081471,0.081471


In [152]:
# [CELL 8] Export CSV Files

balanced_sample.to_csv(
    "balanced_sample.csv",
    index=False
)

feature_comparison.to_csv(
    "feature_comparison.csv",
    index_label="Feature"
)

print("Successfully generated:")
print("1. balanced_sample.csv")
print("2. feature_comparison.csv")

Successfully generated:
1. balanced_sample.csv
2. feature_comparison.csv


In [153]:
# [CELL 9] Verify NumPy-Pandas Statistics Match

numpy_pass_mean = np.mean(pass_data.to_numpy(), axis=0)
numpy_fail_mean = np.mean(fail_data.to_numpy(), axis=0)

pandas_pass_mean = pass_mean.to_numpy()
pandas_fail_mean = fail_mean.to_numpy()

statistics_match = (
    np.allclose(numpy_pass_mean, pandas_pass_mean)
    and np.allclose(numpy_fail_mean, pandas_fail_mean)
)

highest_ranked_feature = feature_comparison[
    "Absolute_Standardized_Mean_Difference"
].idxmax()

print("EXERCISE 2 RESULTS")
print("------------------")

print("Balanced Sample Size:", len(balanced_sample))
print("Selected Features:", selected_features)
print("Highest-Ranked Feature:", highest_ranked_feature)

print(
    "Absolute Standardized Mean Difference:",
    feature_comparison.loc[
        highest_ranked_feature,
        "Absolute_Standardized_Mean_Difference"
    ]
)

print("NumPy-Pandas Statistics Match:", statistics_match)

EXERCISE 2 RESULTS
------------------
Balanced Sample Size: 40
Selected Features: [171, 228, 289, 298, 564]
Highest-Ranked Feature: 171
Absolute Standardized Mean Difference: 0.5761389406394227
NumPy-Pandas Statistics Match: True


# EXERCISE 3: Statistical Robustness and Sensitivity Analysis

In [154]:
# [CELL 1] Import libraries and load SECOM data

import numpy as np
import pandas as pd

SEED_DIGIT = 6
RANDOM_SEED = 6
BOOTSTRAP_REPETITIONS = 200 + SEED_DIGIT

# Load measurement data
X = pd.read_csv(
    "secom.data",
    sep=r"\s+",
    header=None,
    na_values=["?", "NaN"]
)

# Load outcome labels
labels = pd.read_csv(
    "secom_labels.data",
    sep=r"\s+",
    header=None
)

# Create labeled DataFrame
df = X.copy()
df.columns = [f"Feature_{i+1:03d}" for i in range(X.shape[1])]

# Preserve original record identifiers
df.insert(0, "Record_ID", np.arange(len(df)))

# Convert labels into readable outcomes
df["Outcome"] = labels.iloc[:, 0].map({
    1: "Pass",
    -1: "Fail"
})

# Keep only records with valid outcome labels
df = df.dropna(subset=["Outcome"]).copy()

feature_cols = [
    col for col in df.columns
    if col.startswith("Feature_")
]

print("Dataset shape:", df.shape)
print("Pass records:", (df["Outcome"] == "Pass").sum())
print("Fail records:", (df["Outcome"] == "Fail").sum())
print("Bootstrap repetitions:", BOOTSTRAP_REPETITIONS)

Dataset shape: (1567, 592)
Pass records: 104
Fail records: 1463
Bootstrap repetitions: 206


In [155]:
# [CELL 2] Identify usable features

usable_features = []

for feature in feature_cols:
    pass_count = df.loc[
        df["Outcome"] == "Pass", feature
    ].notna().sum()

    fail_count = df.loc[
        df["Outcome"] == "Fail", feature
    ].notna().sum()

    if pass_count >= 10 and fail_count >= 10:
        usable_features.append(feature)

if len(usable_features) < 5:
    raise ValueError("Fewer than five usable features were found.")

rng = np.random.default_rng(RANDOM_SEED)

selected_features = sorted(
    rng.choice(
        usable_features,
        size=5,
        replace=False
    ).tolist()
)

print("Number of usable features:", len(usable_features))
print("Selected features:", selected_features)

Number of usable features: 586
Selected features: ['Feature_203', 'Feature_262', 'Feature_307', 'Feature_318', 'Feature_559']


In [156]:
# [CELL 3] Prepare available-case and imputed datasets

# Available-case data retains missing values
available_data = df[
    ["Record_ID", "Outcome"] + selected_features
].copy()

# Calculate medians using the full dataset
full_dataset_medians = df[selected_features].median()

# Impute missing values
imputed_data = available_data.copy()

imputed_data[selected_features] = (
    imputed_data[selected_features]
    .fillna(full_dataset_medians)
)

print("Full-dataset medians:")
print(full_dataset_medians)

print("\nRemaining missing values after imputation:")
print(imputed_data[selected_features].isna().sum())

Full-dataset medians:
Feature_203    8.4620
Feature_262    0.0000
Feature_307    0.0448
Feature_318    0.1634
Feature_559    0.9727
dtype: float64

Remaining missing values after imputation:
Feature_203    0
Feature_262    0
Feature_307    0
Feature_318    0
Feature_559    0
dtype: int64


In [157]:
# [CELL 4] Calculate mean differences

original_results = []

for feature in selected_features:

    pass_available = available_data.loc[
        available_data["Outcome"] == "Pass", feature
    ].mean()

    fail_available = available_data.loc[
        available_data["Outcome"] == "Fail", feature
    ].mean()

    pass_imputed = imputed_data.loc[
        imputed_data["Outcome"] == "Pass", feature
    ].mean()

    fail_imputed = imputed_data.loc[
        imputed_data["Outcome"] == "Fail", feature
    ].mean()

    original_results.append({
        "Feature": feature,
        "Available_Pass_Mean": pass_available,
        "Available_Fail_Mean": fail_available,
        "Available_Mean_Difference": fail_available - pass_available,
        "Imputed_Pass_Mean": pass_imputed,
        "Imputed_Fail_Mean": fail_imputed,
        "Imputed_Mean_Difference": fail_imputed - pass_imputed
    })

original_results = pd.DataFrame(original_results)

print(original_results.round(6))

       Feature  Available_Pass_Mean  Available_Fail_Mean  \
0  Feature_203            10.904903            10.118003   
1  Feature_262             0.000000             0.000000   
2  Feature_307             0.048941             0.048600   
3  Feature_318             0.161855             0.173396   
4  Feature_559             1.002287             0.994491   

   Available_Mean_Difference  Imputed_Pass_Mean  Imputed_Fail_Mean  \
0                  -0.786900          10.904903          10.110080   
1                   0.000000           0.000000           0.000000   
2                  -0.000341           0.048941           0.048598   
3                   0.011541           0.161855           0.173389   
4                  -0.007795           1.002287           0.994476   

   Imputed_Mean_Difference  
0                -0.794823  
1                 0.000000  
2                -0.000343  
3                 0.011534  
4                -0.007810  


In [158]:
# [CELL 5] Stratified bootstrap function

def run_bootstrap(data, features, repetitions, seed):

    rng = np.random.default_rng(seed)

    pass_data = data[data["Outcome"] == "Pass"]
    fail_data = data[data["Outcome"] == "Fail"]

    bootstrap_results = {
        feature: {
            "Available": [],
            "Imputed": []
        }
        for feature in features
    }

    for repetition in range(repetitions):

        # Sample independently within each outcome group
        pass_sample = pass_data.iloc[
            rng.integers(0, len(pass_data), size=len(pass_data))
        ]

        fail_sample = fail_data.iloc[
            rng.integers(0, len(fail_data), size=len(fail_data))
        ]

        for feature in features:

            # Available-case means
            pass_mean_available = pass_sample[feature].mean()
            fail_mean_available = fail_sample[feature].mean()

            difference_available = (
                fail_mean_available - pass_mean_available
            )

            # Imputed means
            pass_mean_imputed = pass_sample[feature].fillna(
                full_dataset_medians[feature]
            ).mean()

            fail_mean_imputed = fail_sample[feature].fillna(
                full_dataset_medians[feature]
            ).mean()

            difference_imputed = (
                fail_mean_imputed - pass_mean_imputed
            )

            bootstrap_results[feature]["Available"].append(
                difference_available
            )

            bootstrap_results[feature]["Imputed"].append(
                difference_imputed
            )

    return bootstrap_results

In [159]:
# [CELL 6] Run bootstrap analysis

bootstrap_results = run_bootstrap(
    available_data,
    selected_features,
    BOOTSTRAP_REPETITIONS,
    RANDOM_SEED
)

bootstrap_summary = []

for feature in selected_features:

    for strategy in ["Available", "Imputed"]:

        values = np.array(
            bootstrap_results[feature][strategy],
            dtype=float
        )

        values = values[np.isfinite(values)]

        if len(values) == 0:
            raise ValueError(
                f"No valid bootstrap results for {feature}, {strategy}."
            )

        bootstrap_summary.append({
            "Feature": feature,
            "Strategy": strategy,
            "Bootstrap_Mean_Difference": np.mean(values),
            "CI_95_Lower": np.percentile(values, 2.5),
            "CI_95_Upper": np.percentile(values, 97.5),
            "Valid_Repetitions": len(values)
        })

bootstrap_summary = pd.DataFrame(bootstrap_summary)

print(bootstrap_summary.round(6))

       Feature   Strategy  Bootstrap_Mean_Difference  CI_95_Lower  \
0  Feature_203  Available                  -0.711775    -3.290155   
1  Feature_203    Imputed                  -0.719389    -3.302327   
2  Feature_262  Available                   0.000000     0.000000   
3  Feature_262    Imputed                   0.000000     0.000000   
4  Feature_307  Available                  -0.000510    -0.004364   
5  Feature_307    Imputed                  -0.000513    -0.004366   
6  Feature_318  Available                   0.011336    -0.000901   
7  Feature_318    Imputed                   0.011329    -0.000910   
8  Feature_559  Available                  -0.007842    -0.029071   
9  Feature_559    Imputed                  -0.007857    -0.029073   

   CI_95_Upper  Valid_Repetitions  
0     1.233455                206  
1     1.225066                206  
2     0.000000                206  
3     0.000000                206  
4     0.003718                206  
5     0.003715          

In [160]:
# [CELL 7] Compare strategies and rankings

comparison = original_results.copy()

# Calculate absolute change in mean difference
comparison["Absolute_Change"] = (
    comparison["Available_Mean_Difference"]
    - comparison["Imputed_Mean_Difference"]
).abs()

# Determine whether the direction changed
comparison["Difference_Sign_Changed"] = (
    np.sign(comparison["Available_Mean_Difference"])
    != np.sign(comparison["Imputed_Mean_Difference"])
)

# Add bootstrap confidence intervals
for strategy, prefix in [
    ("Available", "Available"),
    ("Imputed", "Imputed")
]:

    summary = bootstrap_summary[
        bootstrap_summary["Strategy"] == strategy
    ].set_index("Feature")

    comparison[f"{prefix}_Bootstrap_Mean"] = (
        summary["Bootstrap_Mean_Difference"]
    )

    comparison[f"{prefix}_CI_Lower"] = summary["CI_95_Lower"]
    comparison[f"{prefix}_CI_Upper"] = summary["CI_95_Upper"]

# Rank features using absolute original mean differences
comparison["Available_Rank"] = (
    comparison["Available_Mean_Difference"]
    .abs()
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison["Imputed_Rank"] = (
    comparison["Imputed_Mean_Difference"]
    .abs()
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison["Ranking_Changed"] = (
    comparison["Available_Rank"]
    != comparison["Imputed_Rank"]
)

comparison = comparison.sort_values(
    "Available_Rank"
).reset_index(drop=True)

print(comparison.round(6))

       Feature  Available_Pass_Mean  Available_Fail_Mean  \
0  Feature_203            10.904903            10.118003   
1  Feature_318             0.161855             0.173396   
2  Feature_559             1.002287             0.994491   
3  Feature_307             0.048941             0.048600   
4  Feature_262             0.000000             0.000000   

   Available_Mean_Difference  Imputed_Pass_Mean  Imputed_Fail_Mean  \
0                  -0.786900          10.904903          10.110080   
1                   0.011541           0.161855           0.173389   
2                  -0.007795           1.002287           0.994476   
3                  -0.000341           0.048941           0.048598   
4                   0.000000           0.000000           0.000000   

   Imputed_Mean_Difference  Absolute_Change  Difference_Sign_Changed  \
0                -0.794823         0.007923                    False   
1                 0.011534         0.000007                    False   
2 

In [161]:
# [CELL 8] Verify reproducibility

bootstrap_results_repeat = run_bootstrap(
    available_data,
    selected_features,
    BOOTSTRAP_REPETITIONS,
    RANDOM_SEED
)

reproducibility_verified = True

for feature in selected_features:
    for strategy in ["Available", "Imputed"]:

        first_run = np.array(
            bootstrap_results[feature][strategy]
        )

        second_run = np.array(
            bootstrap_results_repeat[feature][strategy]
        )

        if not np.array_equal(
            first_run,
            second_run,
            equal_nan=True
        ):
            reproducibility_verified = False

print("Reproducibility Verified:", reproducibility_verified)

Reproducibility Verified: True


In [162]:
# [CELL 9] Export statistical comparison

comparison.to_csv(
    "robustness_comparison.csv",
    index=False
)

# Identify assessment values
largest_change_row = comparison.loc[
    comparison["Absolute_Change"].idxmax()
]

largest_change = largest_change_row["Absolute_Change"]
largest_change_feature = largest_change_row["Feature"]

changed_sign_features = comparison.loc[
    comparison["Difference_Sign_Changed"],
    "Feature"
].tolist()

changed_ranking_features = comparison.loc[
    comparison["Ranking_Changed"],
    "Feature"
].tolist()

highest_ranked_feature = comparison.iloc[0]["Feature"]

# Create interpretation report
report = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
===============================================

Selected Features:
{", ".join(selected_features)}

Bootstrap Repetitions:
{BOOTSTRAP_REPETITIONS}

Largest Change in Mean Difference:
Feature: {largest_change_feature}
Absolute Change: {largest_change:.6f}

Features with Changed Difference Sign:
{changed_sign_features if changed_sign_features else "None"}

Features with Changed Ranking:
{changed_ranking_features if changed_ranking_features else "None"}

Highest-Ranked Feature:
{highest_ranked_feature}

Reproducibility Verified:
{reproducibility_verified}

INTERPRETATION
--------------

The analysis compared available-case analysis and median imputation
using the full dataset.

The mean differences show how Fail and Pass measurements differ
under each preparation strategy. The bootstrap confidence intervals
show the range of differences supported by repeated stratified
sampling.

Features with changed difference signs indicate that the direction
of the group difference depends on the preparation method.
Features with changed rankings indicate that the relative importance
of some measurements also depends on the preparation method.

These results demonstrate why data preparation and sampling
stability should be considered before using statistical findings
to support engineering decisions.
"""

with open("sensitivity_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

print("Files successfully exported:")
print("- robustness_comparison.csv")
print("- sensitivity_report.txt")

Files successfully exported:
- robustness_comparison.csv
- sensitivity_report.txt


In [163]:
# [CELL 10] ASSESSMENT DATA: OBSERVED VALUES

print("\nASSESSMENT DATA")
print("=" * 50)

print("Selected Features:", selected_features)
print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)

print(
    "Largest Change in Mean Difference:",
    round(largest_change, 6),
    "(", largest_change_feature, ")"
)

print("Features with Changed Difference Sign:",
      changed_sign_features)

print("Features with Changed Ranking:",
      changed_ranking_features)

print("Reproducibility Verified:",
      reproducibility_verified)


ASSESSMENT DATA
Selected Features: ['Feature_203', 'Feature_262', 'Feature_307', 'Feature_318', 'Feature_559']
Bootstrap Repetitions: 206
Largest Change in Mean Difference: 0.007923 ( Feature_203 )
Features with Changed Difference Sign: []
Features with Changed Ranking: []
Reproducibility Verified: True
